# 实验 1.2 昇腾香橙派开发板系统制作烧录与基础运行实验

——基于 Orange Pi AI Pro（昇腾 AI 处理器）开发板

> **实验定位**：入门级（难度 1），2 学时（90 分钟）

> 本实验围绕香橙派 AI Pro 开发板，完整走通 "镜像下载 → TF 卡烧录 → 拨码启动 → 系统登录 → 基础测试 → SSH 远程连接" 全流程，帮助你掌握昇腾边缘计算开发板从零到一的环境搭建方法，为后续 AI 推理应用开发奠定坚实基础。

## 实验简介

Orange Pi AI Pro（香橙派 AI Pro）是香橙派与华为联合打造的高性能 AI 开发板，搭载昇腾 AI 处理器，专为边缘计算场景设计。本实验安排以下学习路径：

1. **镜像下载与工具准备**：从官网获取 Ubuntu desktop 镜像与烧录工具
2. **拨码开关设置**：掌握 BOOT1/BOOT2 控制启动设备的方法
3. **TF 卡烧录**：使用 Win32Diskimager / balenaEtcher 将镜像写入 TF 卡
4. **eMMC / SSD 烧录（选做）**：借助 TF 卡系统中转烧录到 eMMC 或 NVMe SSD
5. **上电启动与登录**：通过 HDMI 桌面或调试串口登录 Linux 系统
6. **基础运行测试**：LED 灯、以太网、WIFI 连接与 SSH 远程登录验证

> **运行说明**：本 Notebook 中的 **Markdown 步骤** 描述在 PC 与开发板上应执行的操作；标注 `可在开发板上运行` 的代码单元可在 SSH 登录开发板后执行；标注 `可在云平台运行` 的代码单元可在本地 Jupyter 环境直接运行，用于辅助理解和演示。


---

## 一、实验目的

### 1. 知识目标

熟悉 Orange Pi AI Pro 开发板的硬件组成、主要技术规格和接口布局，了解昇腾 AI 处理器的基本性能指标；理解 BOOT1/BOOT2 拨码开关控制启动设备的原理与系统烧录的基本概念。

### 2. 能力目标

掌握 Linux（Ubuntu）系统镜像的获取方法，能够使用 Win32Diskimager、balenaEtcher 等工具将系统镜像烧录到 TF 卡、eMMC 和 NVMe SSD；掌握开发板的上电启动流程以及通过 HDMI 桌面、调试串口登录 Linux 系统的方法；能够完成板载 LED 灯、以太网、WIFI 连接测试及 SSH 远程登录等基础功能验证。

### 3. 素养目标

养成规范操作硬件设备的习惯，建立对昇腾 AI 边缘计算开发板软硬件生态的整体认知，为后续基于 CANN 的 AI 应用开发实验打下环境基础。


## 二、实验设备与环境

### 1. 硬件设备

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">设备名称</th>
<th style="text-align: left;">规格要求</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">Orange Pi AI Pro 开发板</td>
<td style="text-align: left;">昇腾 AI 处理器，8GB / 16GB LPDDR4X 内存</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">TF 卡</td>
<td style="text-align: left;">32GB 及以上，class10 级或以上（推荐 64GB 以上闪迪等品牌卡）</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">TF 卡读卡器</td>
<td style="text-align: left;">用于在电脑上读写 TF 卡</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">电源适配器</td>
<td style="text-align: left;">Type-C 接口，20V PD-65W</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">HDMI 显示器及连接线</td>
<td style="text-align: left;">HDMI 转 HDMI 连接线</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">USB 鼠标、键盘</td>
<td style="text-align: left;">USB 接口</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">网线</td>
<td style="text-align: left;">百兆或千兆网线，可连接因特网</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">Micro USB 数据线</td>
<td style="text-align: left;">用于调试串口（可选）</td>
</tr>
<tr>
<td style="text-align: left;">9</td>
<td style="text-align: left;">eMMC 模块 / NVMe SSD（选做）</td>
<td style="text-align: left;">M.2 M-Key 2280 规格 NVMe SSD；与开发板匹配的 eMMC 模块</td>
</tr>
<tr>
<td style="text-align: left;">10</td>
<td style="text-align: left;">X64 电脑</td>
<td style="text-align: left;">安装 Windows（用于烧录镜像）或 Ubuntu 22.04</td>
</tr>
</table>

### 2. 软件工具

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">软件名称</th>
<th style="text-align: left;">用途与来源</th>
</tr>
<tr>
<td style="text-align: left;">Linux 系统镜像</td>
<td style="text-align: left;">Ubuntu / openEuler 镜像，从香橙派官网下载页面获取</td>
</tr>
<tr>
<td style="text-align: left;">SD Card Formatter</td>
<td style="text-align: left;">格式化 TF 卡（Windows），下载地址：<code>https://www.sdcard.org/downloads/formatter/</code></td>
</tr>
<tr>
<td style="text-align: left;">Win32Diskimager</td>
<td style="text-align: left;">Windows 下将镜像烧录到 TF 卡</td>
</tr>
<tr>
<td style="text-align: left;">balenaEtcher</td>
<td style="text-align: left;">Windows / Ubuntu 下烧录镜像，下载地址：<code>https://www.balena.io/etcher/</code></td>
</tr>
<tr>
<td style="text-align: left;">MobaXterm / putty</td>
<td style="text-align: left;">串口调试与 SSH 远程登录终端软件</td>
</tr>
</table>

### 3. 实验基本信息

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;">涉及华为技术</td>
<td style="text-align: left;">昇腾 AI 处理器硬件（Ascend 310B4 NPU）</td>
</tr>
<tr>
<td style="text-align: left;">难度级别</td>
<td style="text-align: left;">1（入门）</td>
</tr>
<tr>
<td style="text-align: left;">核心技能</td>
<td style="text-align: left;">熟悉昇腾开发板的系统烧录与基础运行验证</td>
</tr>
<tr>
<td style="text-align: left;">课时数</td>
<td style="text-align: left;">2 学时（90 分钟）</td>
</tr>
<tr>
<td style="text-align: left;">前置知识</td>
<td style="text-align: left;">Linux 基本命令、计算机硬件基础</td>
</tr>
</table>


---

## 三、实验原理

### 1. 开发板简介

Orange Pi AI Pro 开发板是香橙派联合华为精心打造的高性能 AI 开发板，其搭载了昇腾 AI 处理器（4 核 64 位 Arm 处理器 + AI 处理器），可提供 **8 TOPS INT8**（4 TFLOPS FP16）的计算能力，内存提供 8GB 和 16GB 两种版本，可实现图像、视频等多种数据分析与推理计算，广泛用于教育、机器人、无人机等场景。

![Orange Pi AI Pro 开发板外观](images/board_appearance.png)

<div align="center">Orange Pi AI Pro 开发板外观</div>

#### 开发板顶层视图与底层视图

![开发板顶层视图](images/board_top_view.png)

<div align="center">开发板顶层视图</div>

![开发板底层视图（背面可见 BOOT1、BOOT2 拨码开关）](images/board_bottom_view.png)

<div align="center">开发板底层视图（背面可见 BOOT1、BOOT2 拨码开关）</div>

#### 开发板接口详情图

![开发板接口详情图](images/board_interfaces.png)

<div align="center">开发板接口详情图</div>


### 2. 开发板硬件规格

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">规格</th>
</tr>
<tr>
<td style="text-align: left;">昇腾 AI 处理器</td>
<td style="text-align: left;">4 核 64 位 Arm 处理器 + AI 处理器</td>
</tr>
<tr>
<td style="text-align: left;">AI 算力</td>
<td style="text-align: left;">半精度（FP16）：4 TFLOPS；整数精度（INT8）：8 TOPS</td>
</tr>
<tr>
<td style="text-align: left;">内存</td>
<td style="text-align: left;">LPDDR4X，8GB 或 16GB</td>
</tr>
<tr>
<td style="text-align: left;">存储</td>
<td style="text-align: left;">板载 32MB SPI Flash、Micro SD 卡插槽、eMMC 插座、M.2 M-Key 接口（2280 NVMe/SATA SSD）</td>
</tr>
<tr>
<td style="text-align: left;">以太网</td>
<td style="text-align: left;">10/100/1000Mbps，板载 PHY 芯片 RTL8211F</td>
</tr>
<tr>
<td style="text-align: left;">Wi-Fi + 蓝牙</td>
<td style="text-align: left;">2.4G/5G 双频 WIFI，BT4.2</td>
</tr>
<tr>
<td style="text-align: left;">USB</td>
<td style="text-align: left;">2 个 USB3.0 Host 接口、1 个 Type-C（USB3.0）接口</td>
</tr>
<tr>
<td style="text-align: left;">摄像头</td>
<td style="text-align: left;">2 个 MIPI CSI 2 Lane 接口</td>
</tr>
<tr>
<td style="text-align: left;">显示</td>
<td style="text-align: left;">2 个 HDMI 接口、1 个 MIPI DSI 2 Lane 接口</td>
</tr>
<tr>
<td style="text-align: left;">40 pin 扩展口</td>
<td style="text-align: left;">UART / I2C / SPI / PWM / GPIO</td>
</tr>
<tr>
<td style="text-align: left;">按键</td>
<td style="text-align: left;">复位键、关机键、升级按键</td>
</tr>
<tr>
<td style="text-align: left;">拨码开关</td>
<td style="text-align: left;">2 个（BOOT1、BOOT2），控制启动设备</td>
</tr>
<tr>
<td style="text-align: left;">电源</td>
<td style="text-align: left;">Type-C 供电，20V PD-65W 适配器</td>
</tr>
<tr>
<td style="text-align: left;">LED 灯</td>
<td style="text-align: left;">1 个电源指示灯 + 1 个软件可控指示灯</td>
</tr>
<tr>
<td style="text-align: left;">调试串口</td>
<td style="text-align: left;">Micro USB 接口调试串口</td>
</tr>
<tr>
<td style="text-align: left;">操作系统</td>
<td style="text-align: left;">Ubuntu 22.04 和 openEuler 22.03</td>
</tr>
<tr>
<td style="text-align: left;">产品尺寸 / 重量</td>
<td style="text-align: left;">107×68mm / 82g</td>
</tr>
</table>


### 3. 启动介质与拨码开关

开发板支持从 **TF 卡**、**eMMC** 和 **SSD**（NVMe SSD 和 SATA SSD）三种介质启动。具体从哪个设备启动由开发板背面的 **BOOT1** 和 **BOOT2** 两个拨码开关控制。两个拨码开关各有左右两种状态，共 4 种组合，目前使用了其中 3 种：

![开发板背面的 BOOT1、BOOT2 拨码开关](images/boot_switches.jpg)

<div align="center">开发板背面的 BOOT1、BOOT2 拨码开关</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">拨码开关 BOOT1</th>
<th style="text-align: left;">拨码开关 BOOT2</th>
<th style="text-align: left;">对应的启动设备</th>
</tr>
<tr>
<td style="text-align: left;">左</td>
<td style="text-align: left;">左</td>
<td style="text-align: left;">未使用</td>
</tr>
<tr>
<td style="text-align: left;">右</td>
<td style="text-align: left;">左</td>
<td style="text-align: left;">SATA SSD 和 NVMe SSD</td>
</tr>
<tr>
<td style="text-align: left;">左</td>
<td style="text-align: left;">右</td>
<td style="text-align: left;">eMMC</td>
</tr>
<tr>
<td style="text-align: left;"><strong>右</strong></td>
<td style="text-align: left;"><strong>右</strong></td>
<td style="text-align: left;"><strong>TF 卡</strong>（本实验默认）</td>
</tr>
</table>

> SATA SSD 和 NVMe SSD 启动对应的拨码状态相同，两种启动方式通过 M2_TYPE 引脚的电平自动区分。

> ⚠️ **重要**：切换拨码开关后**必须重新拔插电源上下电**才能让新的启动设备选项生效，仅按复位按键重启不会使新的拨码配置生效。


### 4. 系统烧录原理

系统烧录的本质是将完整的操作系统镜像（`.img` 文件）按扇区逐块写入启动介质。

- **烧录 TF 卡**：可借助读卡器在 Windows / Ubuntu 电脑上使用 Win32Diskimager 或 balenaEtcher 直接写入。
- **烧录 eMMC 和 SSD**：开发板没有提供直接从电脑烧录的通道，需要先用 TF 卡启动开发板进入 Linux 系统，再在开发板上运行预装的 balenaEtcher 将镜像写入 eMMC（设备节点 `/dev/mmcblk0`）或 SSD（设备节点 `/dev/nvme0n1` 或 `/dev/sda`），最后调整拨码开关从新介质启动。

```
┌─────────────────────────────────────────────────────────┐
│  烧录流程对比                                            │
├─────────────────────────────────────────────────────────┤
│  TF 卡:  PC ──读卡器──▶ TF 卡（直接烧录）                │
│  eMMC :  PC ──▶ TF 卡 ──▶ 启动开发板 ──▶ balenaEtcher ──▶ eMMC  │
│  SSD  :  PC ──▶ TF 卡 ──▶ 启动开发板 ──▶ balenaEtcher ──▶ SSD   │
└─────────────────────────────────────────────────────────┘
```


---

## 四、实验内容与步骤

本实验共八个任务，建议按顺序完成。若开发板已由实验室预先烧录好操作系统，可从任务六开始，但任务一至五的烧录流程仍需阅读并理解。


### 任务一：下载镜像与准备烧录工具

#### 1.1 下载 Linux 系统镜像

打开香橙派官网 Orange Pi AI Pro 资料下载页面：

`http://www.orangepi.cn/html/hardWare/computerAndMicrocontrollers/service-and-support/Orange-Pi-AIpro.html`

![Orange Pi AI Pro 官方资料下载页面](images/download_page.jpg)

<div align="center">Orange Pi AI Pro 官方资料下载页面</div>

Ubuntu 镜像分为两种：

![minimal 镜像与 desktop 镜像对比](images/image_types.jpg)

<div align="center">minimal 镜像与 desktop 镜像对比</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">镜像类型</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>minimal</strong></td>
<td style="text-align: left;">仅含最基础功能，未预装桌面、CANN 和 AI 示例代码。适合想自己从头定制的开发者</td>
</tr>
<tr>
<td style="text-align: left;"><strong>desktop</strong></td>
<td style="text-align: left;">预装了 Linux 桌面、CANN、AI 示例代码和一系列测试程序。<strong>本实验选用 desktop 镜像</strong></td>
</tr>
</table>

> 镜像为 `.img.xz` 压缩格式，balenaEtcher 可直接识别，**无需手动解压**；使用 Win32Diskimager 则需先解压得到 `.img` 文件。

#### 1.2 准备烧录工具

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">平台</th>
<th style="text-align: left;">工具</th>
<th style="text-align: left;">用途</th>
</tr>
<tr>
<td style="text-align: left;">Windows</td>
<td style="text-align: left;">SD Card Formatter</td>
<td style="text-align: left;">格式化 TF 卡</td>
</tr>
<tr>
<td style="text-align: left;">Windows</td>
<td style="text-align: left;">Win32Diskimager</td>
<td style="text-align: left;">将 <code>.img</code> 镜像写入 TF 卡</td>
</tr>
<tr>
<td style="text-align: left;">Windows / Ubuntu</td>
<td style="text-align: left;">balenaEtcher</td>
<td style="text-align: left;">直接烧录 <code>.img.xz</code> 压缩包</td>
</tr>
</table>

#### 1.3 准备硬件配件

![TF 卡（≥32GB，class10）与读卡器](images/acc_tf_card.jpg) &nbsp;&nbsp; ![TF 卡（≥32GB，class10）与读卡器](images/acc_card_reader.jpg)

<div align="center">TF 卡（≥32GB，class10）与读卡器</div>

![Type-C 20V PD-65W 电源适配器与 HDMI 连接线](images/acc_power_adapter.jpg) &nbsp;&nbsp; ![Type-C 20V PD-65W 电源适配器与 HDMI 连接线](images/acc_hdmi_cable.jpg)

<div align="center">Type-C 20V PD-65W 电源适配器与 HDMI 连接线</div>

![网线与 Micro USB 数据线（调试串口用）](images/acc_ethernet_cable.jpg) &nbsp;&nbsp; ![网线与 Micro USB 数据线（调试串口用）](images/acc_micro_usb_cable.jpg)

<div align="center">网线与 Micro USB 数据线（调试串口用）</div>


### 任务二：设置拨码开关选择启动设备

根据实验使用的启动介质，按照下表的对应关系设置开发板背面的 BOOT1、BOOT2 拨码开关。**本实验主流程使用 TF 卡启动**，即将 BOOT1、BOOT2 均拨到右侧（右-右）。

![拨码开关位置（开发板背面）](images/boot_switches.jpg)

<div align="center">拨码开关位置（开发板背面）</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">BOOT1</th>
<th style="text-align: left;">BOOT2</th>
<th style="text-align: left;">启动设备</th>
<th style="text-align: left;">本实验</th>
</tr>
<tr>
<td style="text-align: left;">左</td>
<td style="text-align: left;">左</td>
<td style="text-align: left;">未使用</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">右</td>
<td style="text-align: left;">左</td>
<td style="text-align: left;">SATA/NVMe SSD</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">左</td>
<td style="text-align: left;">右</td>
<td style="text-align: left;">eMMC</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"><strong>右</strong></td>
<td style="text-align: left;"><strong>右</strong></td>
<td style="text-align: left;"><strong>TF 卡</strong></td>
<td style="text-align: left;"><strong>✓</strong></td>
</tr>
</table>

> 设置完成后**插拔一次电源**使其生效。


### 任务三：烧录 Linux 镜像到 TF 卡

#### 方法一：Windows 下使用 Win32Diskimager 烧录

**步骤 1：格式化 TF 卡**

将 TF 卡插入读卡器并连接电脑 USB 接口。打开 SD Card Formatter，确认 "Select card" 一栏显示 TF 卡盘符：

![SD Card Formatter 选择 TF 卡](images/sd_formatter_select.jpg)

<div align="center">SD Card Formatter 选择 TF 卡</div>

点击 "Format"，在弹出的警告框中选择 "是(Y)" 开始格式化：

![格式化警告框](images/sd_formatter_warning.jpg)

<div align="center">格式化警告框</div>

![格式化完成提示](images/sd_formatter_done.jpg)

<div align="center">格式化完成提示</div>

**步骤 2：写入镜像**

打开 Win32Diskimager，选择解压后的镜像文件（`.img`）路径，确认 "设备" 一栏显示的盘符与 TF 卡一致，点击 "写入" 开始烧录：

![Win32Diskimager 烧录界面](images/win32diskimager.jpg)

<div align="center">Win32Diskimager 烧录界面</div>

镜像写入完成后点击 "退出"，即可拔出 TF 卡插入开发板启动。

> ⚠️ 烧录完成后若 Windows 弹出 "是否格式化磁盘" 提示，请选择 **"取消"**，切勿选择 "格式化磁盘"，否则会将已烧录的镜像破坏。


#### 方法二：Windows 下使用 balenaEtcher 烧录

**步骤 1：下载 balenaEtcher**

前往 `https://www.balena.io/etcher/` 下载 Portable 版本（无需安装，双击打开即可使用）：

![balenaEtcher 下载页面](images/etcher_download.jpg)

<div align="center">balenaEtcher 下载页面</div>

![选择 Portable 版本](images/etcher_portable.jpg)

<div align="center">选择 Portable 版本</div>

> 若打开时提示错误，请右键选择 "以管理员身份运行"。

**步骤 2：烧录镜像**

打开 balenaEtcher，依次操作：选择镜像文件 → 选择 TF 卡盘符 → 点击 Flash：

![balenaEtcher 主界面](images/etcher_interface.jpg)

<div align="center">balenaEtcher 主界面</div>

![选择镜像文件与 TF 卡](images/etcher_select_image.jpg)

<div align="center">选择镜像文件与 TF 卡</div>

烧录过程中进度条显示**紫色**表示正在写入：

![烧录中（紫色进度条）](images/etcher_flashing.jpg)

<div align="center">烧录中（紫色进度条）</div>

写入完成后 balenaEtcher 会自动校验，进度条变**绿色**：

![校验中（绿色进度条）](images/etcher_validating.jpg)

<div align="center">校验中（绿色进度条）</div>

![烧录成功（绿色指示图标）](images/etcher_success.jpg)

<div align="center">烧录成功（绿色指示图标）</div>

> 💡 balenaEtcher 可直接烧录 `.img.xz` 压缩包，无需手动解压。


#### 方法三：Ubuntu PC 下使用 balenaEtcher 烧录

在 Ubuntu 电脑上下载 Linux 版本的 balenaEtcher（AppImage 格式）：

![下载 Linux 版本](images/etcher_linux_download.jpg)

<div align="center">下载 Linux 版本</div>

在图形界面中双击 `balenaEtcher-x.x.x-x64.AppImage` 打开软件：

![双击 AppImage 打开 balenaEtcher](images/etcher_appimage.jpg)

<div align="center">双击 AppImage 打开 balenaEtcher</div>

依次选择镜像文件、TF 卡盘符，点击 Flash 开始烧录：

![选择镜像与 TF 卡](images/etcher_ubuntu_select.jpg)

<div align="center">选择镜像与 TF 卡</div>

![烧录中](images/etcher_ubuntu_flashing.jpg)

<div align="center">烧录中</div>

![校验中](images/etcher_ubuntu_validating.jpg)

<div align="center">校验中</div>

![烧录成功](images/etcher_ubuntu_success.jpg)

<div align="center">烧录成功</div>


#### 烧录结果验证（可在云平台运行）

以下代码模拟烧录后的验证逻辑，帮助理解烧录校验的原理。在云平台 Jupyter 中可直接运行：


In [ ]:
# 可在云平台运行 —— 模拟镜像烧录校验流程
import os, hashlib

def simulate_burn_verification(image_name, target_device, image_size_mb=4096):
    """模拟烧录与校验流程"""
    print(f"{'='*50}")
    print(f"  镜像烧录模拟器")
    print(f"{'='*50}")
    print(f"  镜像文件 : {image_name}")
    print(f"  目标设备 : {target_device}")
    print(f"  镜像大小 : {image_size_mb} MB")
    print(f"{'-'*50}")
    
    steps = [
        ("1. 挂载目标设备",      "OK"),
        ("2. 擦除原有分区表",    "OK"),
        ("3. 逐扇区写入镜像",    "OK"),
        ("4. 写入引导扇区",      "OK"),
        ("5. 同步缓冲区 (sync)", "OK"),
        ("6. 读取回写数据校验",  "OK"),
        ("7. 校验 SHA256 校验和", "OK"),
    ]
    for desc, status in steps:
        print(f"  [{status}] {desc}")
    
    print(f"{'-'*50}")
    print(f"  烧录结果: 成功 ✓")
    print(f"  提示: 拔出 TF 卡前请先安全弹出读卡器")
    print(f"{'='*50}")

simulate_burn_verification(
    image_name="opiaipro_ubuntu22.04_desktop_aarch64.img.xz",
    target_device="TF 卡 (/dev/sdX)",
    image_size_mb=4096
)


### 任务四：烧录 Linux 镜像到 eMMC（选做）

> eMMC 烧录需要借助 TF 卡中转，请先完成任务三。

**步骤 1：安装 eMMC 模块**

将与开发板 eMMC 接口匹配的 eMMC 模块安装到开发板上：

![eMMC 模块与安装到开发板](images/acc_emmc_module.jpg) &nbsp;&nbsp; ![eMMC 模块与安装到开发板](images/acc_emmc_installed.jpg)

<div align="center">eMMC 模块与安装到开发板</div>

![eMMC 模块安装到开发板背面](images/emmc_install.jpg)

<div align="center">eMMC 模块安装到开发板背面</div>

**步骤 2：从 TF 卡启动并确认 eMMC 已识别**

先用 TF 卡启动开发板进入 Linux 系统，在 root 用户下执行 `fdisk -l` 确认 eMMC 已被识别：

```bash
# 在开发板终端执行（root 用户）
fdisk -l
# 预期输出包含：
# Disk /dev/mmcblk0: 28.91 GiB, 31037849600 bytes, 60620800 sectors
```

**步骤 3：使用 balenaEtcher 烧录镜像到 eMMC**

将镜像压缩包上传到 TF 卡 Linux 系统中，打开预装的 balenaEtcher：

![开发板中打开 balenaEtcher](images/etcher_open.jpg) &nbsp;&nbsp; ![开发板中打开 balenaEtcher](images/etcher_board_interface.jpg)

<div align="center">开发板中打开 balenaEtcher</div>

![Flash from file 选择镜像](images/etcher_flash_from_file.jpg)

<div align="center">Flash from file 选择镜像</div>

> 若提示没有权限，先执行 `sudo chmod 777 镜像文件名` 添加权限。

![Select target 选择 eMMC 对应的 /dev/mmcblk0](images/etcher_select_target.jpg) &nbsp;&nbsp; ![Select target 选择 eMMC 对应的 /dev/mmcblk0](images/etcher_select_mmcblk0.jpg)

<div align="center">Select target 选择 eMMC 对应的 /dev/mmcblk0</div>

![点击 Flash! 开始烧录](images/etcher_flash_btn.jpg)

<div align="center">点击 Flash! 开始烧录</div>

![输入系统密码 Mind@123](images/etcher_password.jpg)

<div align="center">输入系统密码 Mind@123</div>

![烧录进行中](images/emmc_flashing.jpg)

<div align="center">烧录进行中</div>

![eMMC 烧录完成](images/emmc_success.jpg)

<div align="center">eMMC 烧录完成</div>

**步骤 4：切换拨码开关从 eMMC 启动**

烧录完成后关闭系统，拔出 TF 卡并断开电源，将拨码开关拨到 eMMC 启动位置（**BOOT1 左、BOOT2 右**），重新上电即可从 eMMC 启动。


### 任务五：烧录 Linux 镜像到 NVMe SSD（选做）

> SSD 烧录同样需要借助 TF 卡中转。目前实测仅三星品牌的 NVMe SSD 能稳定运行 Linux 系统。

**步骤 1：安装 NVMe SSD**

准备一块 2280 规格 NVMe SSD（M.2 插槽支持 PCIe3.0 x4），插入开发板 M.2 接口并固定：

![NVMe SSD 与安装到开发板 M.2 接口](images/acc_nvme_ssd.jpg) &nbsp;&nbsp; ![NVMe SSD 与安装到开发板 M.2 接口](images/ssd_install.jpg)

<div align="center">NVMe SSD 与安装到开发板 M.2 接口</div>

**步骤 2：从 TF 卡启动并确认 SSD 已识别**

```bash
# 在开发板终端执行
sudo fdisk -l | grep "nvme0n1"
# 预期输出：
# Disk /dev/nvme0n1: 238.47 GiB, 256060514304 bytes, 500118192 sectors
```

**步骤 3：使用 balenaEtcher 烧录镜像到 SSD**

将镜像上传到 TF 卡 Linux 系统，打开 balenaEtcher，选择镜像文件后点击 Select target，点击 **Show 1 hidden** 展开隐藏设备：

![点击 Show 1 hidden 展开隐藏设备](images/etcher_show_hidden.jpg)

<div align="center">点击 Show 1 hidden 展开隐藏设备</div>

![选择 SSD 对应的设备](images/etcher_select_ssd.jpg)

<div align="center">选择 SSD 对应的设备</div>

![点击 Flash!](images/ssd_flash_btn.jpg)

<div align="center">点击 Flash!</div>

![选择 Yes, I'm sure 确认](images/etcher_yes_sure.jpg)

<div align="center">选择 Yes, I'm sure 确认</div>

![输入密码 Mind@123](images/etcher_password.jpg)

<div align="center">输入密码 Mind@123</div>

![SSD 烧录进行中](images/ssd_flashing.jpg)

<div align="center">SSD 烧录进行中</div>

![SSD 烧录完成](images/ssd_success.jpg)

<div align="center">SSD 烧录完成</div>

**步骤 4：切换拨码开关从 SSD 启动**

烧录完成后关机，拔出 TF 卡并断开电源，将拨码开关拨到 SSD 启动位置（**BOOT1 右、BOOT2 左**），重新上电即可从 SSD 启动。


### 任务六：上电启动与系统登录

#### 6.1 连接外设与上电

1. 将烧录好镜像的 TF 卡插入开发板 TF 卡槽
2. 用 HDMI 线将开发板 **HDMI0** 接口（默认主屏）连接到 HDMI 显示器
3. 接上 USB 鼠标和键盘
4. 插入网线（连接路由器或交换机）
5. 连接 20V PD-65W 的 Type-C 电源并打开电源开关

![HDMI0 接口连接显示器](images/startup_hdmi_connect.jpg)

<div align="center">HDMI0 接口连接显示器</div>

![Type-C 电源接口位置](images/startup_power_port.jpg)

<div align="center">Type-C 电源接口位置</div>

#### 6.2 登录 Linux 桌面

上电后等待一段时间，HDMI 显示器即可显示 Linux 系统桌面。桌面系统会**自动登录**，无需输入账号和密码：

![Ubuntu 桌面系统（自动登录）](images/linux_desktop.jpg)

<div align="center">Ubuntu 桌面系统（自动登录）</div>

#### 6.3 默认登录账号与密码

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">账号</th>
<th style="text-align: left;">密码</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>root</code></td>
<td style="text-align: left;"><code>Mind@123</code></td>
<td style="text-align: left;">管理员账户，SSH 默认允许登录</td>
</tr>
<tr>
<td style="text-align: left;"><code>HwHiAiUser</code></td>
<td style="text-align: left;"><code>Mind@123</code></td>
<td style="text-align: left;">普通用户，HDMI 桌面默认登录用户</td>
</tr>
</table>

> ⚠️ 只要使用的是 Orange Pi 官方提供的 Linux 镜像，输入密码提示错误或 SSH 连接异常时，**不要怀疑密码不对**，而应排查网络连通性、IP 地址等其他原因。


### 任务七：通过调试串口登录（可选）

开发板默认使用 uart0 作为调试串口，有两种使用方式：

#### 方式一：通过 Micro USB 接口（推荐）

uart0 经 CH343P 芯片引出到 Micro USB 接口，只需一根 Micro USB 数据线即可连接：

![Micro USB 调试串口接口](images/uart_micro_usb.jpg)

<div align="center">Micro USB 调试串口接口</div>

#### 方式二：通过 40 pin 接口

uart0 的 TX/RX 引脚接到 40 pin 扩展接口的 8 号和 10 号引脚，需要 3.3V USB 转 TTL 模块：

![40 pin 接口中的 uart0 引脚](images/uart_40pin.jpg)

<div align="center">40 pin 接口中的 uart0 引脚</div>

![USB 转 TTL 模块与连接示意图（GND-GND, RX-TX, TX-RX 交叉连接）](images/usb_ttl.jpg) &nbsp;&nbsp; ![USB 转 TTL 模块与连接示意图（GND-GND, RX-TX, TX-RX 交叉连接）](images/uart_connection.jpg)

<div align="center">USB 转 TTL 模块与连接示意图（GND-GND, RX-TX, TX-RX 交叉连接）</div>

> ⚠️ 两种方式只能二选一，不能同时使用。

#### 串口参数设置

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
</tr>
<tr>
<td style="text-align: left;">波特率</td>
<td style="text-align: left;">115200</td>
</tr>
<tr>
<td style="text-align: left;">流控</td>
<td style="text-align: left;">None</td>
</tr>
</table>

#### Windows 下使用 MobaXterm

![下载 MobaXterm Home 版本](images/mobaxterm_download.jpg) &nbsp;&nbsp; ![下载 MobaXterm Home 版本](images/mobaxterm_home.jpg)

<div align="center">下载 MobaXterm Home 版本</div>

![选择 Portable 便携版](images/mobaxterm_portable.jpg)

<div align="center">选择 Portable 便携版</div>

![打开 MobaXterm](images/mobaxterm_open.jpg)

<div align="center">打开 MobaXterm</div>

![设置串口连接：选择端口号与波特率 115200](images/mobaxterm_serial_setup.jpg)

<div align="center">设置串口连接：选择端口号与波特率 115200</div>

![串口终端输出系统 Log 信息](images/mobaxterm_serial_output.jpg)

<div align="center">串口终端输出系统 Log 信息</div>

#### Ubuntu 下使用 putty

```bash
# 在 Ubuntu PC 上安装 putty
sudo apt-get update
sudo apt-get install -y putty
# 运行 putty（需要 sudo 权限）
sudo putty
```

![putty 主界面与串口设置界面](images/putty_main.jpg) &nbsp;&nbsp; ![putty 主界面与串口设置界面](images/putty_serial.jpg)

<div align="center">putty 主界面与串口设置界面</div>

![设置串口参数：Speed 115200, Flow control None](images/putty_params.jpg)

<div align="center">设置串口参数：Speed 115200, Flow control None</div>

![返回 Session 选择 Serial 连接，点击 Open 后可看到系统 Log](images/putty_session.jpg) &nbsp;&nbsp; ![返回 Session 选择 Serial 连接，点击 Open 后可看到系统 Log](images/putty_log.jpg)

<div align="center">返回 Session 选择 Serial 连接，点击 Open 后可看到系统 Log</div>


### 任务八：基础运行测试

#### 8.1 板载 LED 灯检查

开发板上有两个绿色 LED 灯：

![靠近关机按键的绿灯：电源指示灯（硬件控制，上电即亮）](images/led_power.jpg)

<div align="center">靠近关机按键的绿灯：电源指示灯（硬件控制，上电即亮）</div>

![MIPI LCD 与 CAMERA0 之间的绿灯：由 GPIO4_19 控制（内核启动后点亮）](images/led_gpio.jpg)

<div align="center">MIPI LCD 与 CAMERA0 之间的绿灯：由 GPIO4_19 控制（内核启动后点亮）</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">LED 灯</th>
<th style="text-align: left;">控制方式</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">靠近关机按键的绿灯</td>
<td style="text-align: left;">硬件控制</td>
<td style="text-align: left;">电源指示灯，接入 Type-C 电源上电后即点亮</td>
</tr>
<tr>
<td style="text-align: left;">MIPI LCD 与 CAMERA0 之间的绿灯</td>
<td style="text-align: left;">GPIO4_19（软件可控）</td>
<td style="text-align: left;">内核启动后由 DTS 点亮，看到此灯亮说明 Linux 内核已启动</td>
</tr>
</table>


#### 8.2 以太网连接测试

将网线一端插入开发板以太网接口，另一端接入交换机或路由器。系统启动后会通过 DHCP 自动分配 IP 地址。

![以太网口连接网线](images/ethernet_connect.jpg)

<div align="center">以太网口连接网线</div>

**在开发板终端执行以下命令查看 IP 地址：**

```bash
# 查看以太网口 IP 地址
ip a s eth0
```

预期输出（关键信息）：
```
3: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 ...
    link/ether 2c:52:af:89:11:11 brd ff:ff:ff:ff:ff:ff
    inet 192.168.2.100/24 brd 192.168.2.255 scope global dynamic noprefixroute eth0
```

**测试网络连通性：**

```bash
# ping 测试（Ctrl+C 中断）
ping www.baidu.com -I eth0
```

预期输出：
```
PING www.a.shifen.com (183.2.172.185) from 192.168.2.100 eth0: 56(84) bytes of data.
64 bytes from 183.2.172.185: icmp_seq=1 ttl=52 time=10.0 ms
64 bytes from 183.2.172.185: icmp_seq=2 ttl=52 time=9.77 ms
^C
--- www.a.shifen.com ping statistics ---
4 packets transmitted, 4 received, 0% packet loss
```

> `0% packet loss` 表示网络连接正常。


##### 以太网测试模拟（可在云平台运行）

以下代码模拟开发板上的网络测试命令输出，帮助理解测试流程：


In [ ]:
# 可在云平台运行 —— 模拟开发板网络测试输出
import random, time

def simulate_ip_addr(interface="eth0"):
    """模拟 ip a s eth0 命令输出"""
    print(f"(base) HwHiAiUser@orangepiaipro:~$ ip a s {interface}")
    print(f"3: {interface}: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000")
    print(f"    link/ether 2c:52:af:89:11:11 brd ff:ff:ff:ff:ff:ff")
    print(f"    inet 192.168.2.100/24 brd 192.168.2.255 scope global dynamic noprefixroute {interface}")
    print(f"       valid_lft 42077sec preferred_lft 42077sec")
    print(f"    inet6 fe80::913d:474c:4834:a1a4/64 scope link noprefixroute")
    print(f"       valid_lft forever preferred_lft forever")

def simulate_ping(host="www.baidu.com", interface="eth0", count=4):
    """模拟 ping 命令输出"""
    print(f"\n(base) HwHiAiUser@orangepiaipro:~$ ping {host} -I {interface}")
    print(f"PING {host} (183.2.172.185) from 192.168.2.100 {interface}: 56(84) bytes of data.")
    for i in range(1, count + 1):
        rtt = round(random.uniform(9.5, 10.5), 2)
        print(f"64 bytes from 183.2.172.185: icmp_seq={i} ttl=52 time={rtt} ms")
    print("^C")
    print(f"--- {host} ping statistics ---")
    print(f"{count} packets transmitted, {count} received, 0% packet loss, time 3004ms")
    print(f"rtt min/avg/max/mdev = 9.770/9.931/10.065/0.105 ms")

simulate_ip_addr("eth0")
simulate_ping("www.baidu.com", "eth0")


#### 8.3 WIFI 连接测试

> ⚠️ 不要通过修改 `/etc/network/interfaces` 配置文件的方式连接 WIFI，应使用 `nmcli` 或 `nmtui`。

**方法一：通过 nmcli 命令连接**

```bash
# 扫描周围 WIFI 热点
nmcli dev wifi

# 连接指定 WIFI 热点（替换 wifi_name 和 wifi_passwd）
sudo nmcli dev wifi connect wifi_name password wifi_passwd
# 预期输出：Device 'wlan0' successfully activated

# 查看 WIFI 的 IP 地址
ip a s wlan0

# 测试 WIFI 连通性
ping www.orangepi.org -I wlan0
```

**方法二：通过 nmtui 图形化界面连接**

```bash
sudo nmtui
```

![nmtui 主界面](images/nmtui_main.jpg)

<div align="center">nmtui 主界面</div>

![选择 Activate a connection](images/nmtui_activate.jpg)

<div align="center">选择 Activate a connection</div>

![WIFI 热点列表](images/nmtui_wifi_list.jpg)

<div align="center">WIFI 热点列表</div>

![选择要连接的 WIFI 热点](images/nmtui_select_wifi.jpg)

<div align="center">选择要连接的 WIFI 热点</div>

![输入 WIFI 密码](images/nmtui_password.jpg)

<div align="center">输入 WIFI 密码</div>

![连接成功（热点名前显示 * 号）](images/nmtui_connected.jpg)

<div align="center">连接成功（热点名前显示 * 号）</div>

**方法三：通过桌面图形界面连接**

![点击桌面右上角网络图标](images/desktop_network_icon.jpg)

<div align="center">点击桌面右上角网络图标</div>

![选择 More networks 查看所有热点](images/desktop_more_networks.jpg)

<div align="center">选择 More networks 查看所有热点</div>

![输入 WIFI 密码并点击 Connect](images/desktop_wifi_password.jpg)

<div align="center">输入 WIFI 密码并点击 Connect</div>

![WIFI 连接成功](images/desktop_wifi_connected.jpg)

<div align="center">WIFI 连接成功</div>

![打开浏览器验证可上网](images/desktop_browser.jpg)

<div align="center">打开浏览器验证可上网</div>

#### WIFI 蓝牙天线安装注意事项

![WIFI 蓝牙天线](images/wifi_antenna.jpg)

<div align="center">WIFI 蓝牙天线</div>

![正确安装方式](images/wifi_antenna_correct.jpg)

<div align="center">正确安装方式</div>

![错误安装方式（天线贴到背面可能烧坏开发板）](images/wifi_antenna_wrong.jpg)

<div align="center">错误安装方式（天线贴到背面可能烧坏开发板）</div>

> ⚠️ 安装好 WIFI 蓝牙天线后，不要将天线贴到开发板背面，天线上的导电布也不能挨着开发板，否则可能烧坏开发板。


#### 8.4 SSH 远程登录

Linux 系统默认开启 SSH 远程登录且允许 root 用户登录。先确保以太网或 WIFI 已连接并获取开发板 IP 地址。

**Ubuntu 下 SSH 登录：**

```bash
# 在 Ubuntu PC 终端执行（IP 替换为开发板实际地址）
ssh root@192.168.2.xxx
# 输入默认密码：Mind@123
```

![Ubuntu 下 SSH 登录成功](images/ssh_ubuntu_success.jpg)

<div align="center">Ubuntu 下 SSH 登录成功</div>

**Windows 下使用 MobaXterm 登录：**

1. 打开 Session，选择 SSH 类型
2. 在 Remote host 中输入开发板 IP 地址
3. 在 Specify username 中输入 `root` 或 `HwHiAiUser`
4. 点击 OK，输入密码 `Mind@123`

![MobaXterm 新建 SSH 会话设置](images/mobaxterm_ssh_setup.jpg)

<div align="center">MobaXterm 新建 SSH 会话设置</div>

![输入密码 Mind@123](images/mobaxterm_ssh_password.jpg)

<div align="center">输入密码 Mind@123</div>

![Windows 下 SSH 登录成功](images/ssh_windows_success.jpg)

<div align="center">Windows 下 SSH 登录成功</div>

> 💡 输入密码时屏幕不会显示任何字符，属于正常现象，输入完成后直接回车即可。


##### SSH 连接流程模拟（可在云平台运行）


In [ ]:
# 可在云平台运行 —— 模拟 SSH 远程登录开发板流程
def simulate_ssh_login(ip="192.168.2.100", username="HwHiAiUser", password="Mind@123"):
    print("+" + "-"*58 + "+")
    print("|  SSH 远程登录模拟" + " "*40 + "|")
    print("+" + "-"*58 + "+")
    print(f"|  本地终端 -> ssh {username}@{ip}" + " "*max(0, 58-20-len(username)-len(ip)) + "|")
    print(f"|  状态     -> 正在连接..." + " "*36 + "|")
    print(f"|  提示     -> 是否信任主机指纹? (yes/no) yes" + " "*22 + "|")
    print(f"|  提示     -> {ip}'s password: ******** (输入时无回显)" + " "*max(0, 58-30-len(ip)) + "|")
    print(f"|  状态     -> 认证成功" + " "*44 + "|")
    print("+" + "-"*58 + "+")
    print()
    print("Welcome to Orange Pi AI Pro")
    print("System information as of time:")
    print(f"  OS     : Ubuntu 22.04.3 LTS (aarch64)")
    print(f"  Kernel : Linux 5.10.0")
    print(f"  User   : {username}")
    print(f"  Host   : orangepiaipro")
    print()
    print(f"(base) {username}@orangepiaipro:~$ npu-smi info")
    print("+--------------------------------------------------------------------------------+")
    print("| npu-smi 23.0.0                            Version: 23.0.0                                 |")
    print("+----------------------+---------------+--------------------------------------------+|")
    print("| NPU   Name            | Health        | Power(W)    Temp(C)         Hugepages-Usage   |")
    print("+======================+===============+============================================+|")
    print("| 0     AI Pro          | OK            | 8.5         42              0/0               |")
    print("+======================+===============+============================================+|")
    print()
    print("=> SSH 远程连接建立成功，可远程操作开发板")

simulate_ssh_login()


---

## 五、实验现象与结果记录

请根据实际实验过程，将各环节的观察结果记录于下表中：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">实验环节</th>
<th style="text-align: left;">预期现象</th>
<th style="text-align: left;">实际结果（学生填写）</th>
</tr>
<tr>
<td style="text-align: left;">TF 卡烧录</td>
<td style="text-align: left;">烧录工具提示写入完成、校验通过</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">拨码开关设置</td>
<td style="text-align: left;">BOOT1 右、BOOT2 右（TF 卡启动）</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">上电启动</td>
<td style="text-align: left;">HDMI 显示器出现登录界面并自动进入桌面</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">LED 灯检查</td>
<td style="text-align: left;">电源指示灯点亮；软件可控绿灯点亮（内核已启动）</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">以太网测试</td>
<td style="text-align: left;">eth0 获得 IP 地址，ping 通外网，0% 丢包</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">WIFI 测试</td>
<td style="text-align: left;">wlan0 连接热点成功，ping 通外网</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">SSH 远程登录</td>
<td style="text-align: left;">使用 root / HwHiAiUser 账号成功登录开发板</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">eMMC / SSD 烧录（选做）</td>
<td style="text-align: left;">从新介质启动成功，系统运行正常</td>
<td style="text-align: left;"></td>
</tr>
</table>

### 实验结果分析

1. **系统烧录与启动**：成功将 Ubuntu 22.04 desktop 镜像烧录至 TF 卡，拨码开关设置为右-右（TF 卡启动），开发板上电后正常启动并显示 Linux 桌面。
2. **基础功能验证**：板载 LED 灯正常点亮（电源灯 + 内核启动灯），以太网口通过 DHCP 获取 IP 地址并 ping 通外网，WIFI 可连接热点并访问互联网。
3. **远程登录**：通过 SSH 成功远程登录开发板，`npu-smi info` 显示 NPU 状态 Health: OK，表明昇腾 AI 处理器工作正常。

以上结果表明，Orange Pi AI Pro 开发板的基础软硬件环境配置正确，可作为后续昇腾 AI 推理应用开发与部署的实验平台。


---

## 六、常见问题与注意事项

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">问题现象</th>
<th style="text-align: left;">原因与解决办法</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">TF 卡烧录失败或系统不稳定</td>
<td style="text-align: left;">必须使用 class10 级或以上的高速卡，容量 ≥32GB，推荐闪迪等品牌</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">烧录后 Windows 提示 "是否格式化磁盘"</td>
<td style="text-align: left;">正常现象，TF 卡已是 Linux 分区，选择 <strong>"取消"</strong>，切勿格式化</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">切换拨码开关后不生效</td>
<td style="text-align: left;">必须重新<strong>拔插电源</strong>上下电，按复位键无效</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">balenaEtcher 打开镜像提示没有权限</td>
<td style="text-align: left;">执行 <code>sudo chmod 777 镜像文件名</code> 添加权限</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">上电后无法进入系统</td>
<td style="text-align: left;">确认拨码开关在 TF 卡启动位置（右-右）并重新上下电；TF 卡烧录失败则重新烧录</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">调试串口无输出</td>
<td style="text-align: left;">波特率必须为 115200；40 pin 方式与 Micro USB 方式只能二选一</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">WIFI 无法连接</td>
<td style="text-align: left;">不要修改 <code>/etc/network/interfaces</code>，应使用 <code>nmcli</code> 或 <code>nmtui</code></td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">SSH 连接被拒绝</td>
<td style="text-align: left;">排查网络连通性、IP 地址是否正确，不要怀疑默认密码 Mind@123</td>
</tr>
<tr>
<td style="text-align: left;">9</td>
<td style="text-align: left;">NVMe SSD 无法启动</td>
<td style="text-align: left;">目前仅三星品牌 NVMe SSD 能稳定运行，非三星品牌需等后续软件更新</td>
</tr>
<tr>
<td style="text-align: left;">10</td>
<td style="text-align: left;">天线安装后开发板异常</td>
<td style="text-align: left;">不要将 WIFI 天线贴到开发板背面，导电布不能挨着开发板</td>
</tr>
</table>


---

## 七、拓展实验与思考题

### 一、选择题

**1. Orange Pi AI Pro 开发板的拨码开关 BOOT1=右、BOOT2=右 时，开发板从哪种介质启动？**

A. eMMC　　B. TF 卡　　C. NVMe SSD　　D. SATA SSD

**2. 烧录完成后 Windows 弹出 "是否格式化磁盘" 提示，正确的操作是？**

A. 点击"格式化磁盘"　　B. 点击"取消"　　C. 拔出 TF 卡重新烧录　　D. 重启电脑

**3. 切换拨码开关后，以下哪种操作能让新配置生效？**

A. 按复位按键　　B. 等待 10 秒　　C. 重新拔插电源上下电　　D. 执行 reboot 命令

**4. Orange Pi AI Pro 默认的 SSH 登录密码是？**

A. root　　B. 123456　　C. orangepi　　D. Mind@123

**5. 关于烧录 eMMC 和 SSD，以下说法正确的是？**

A. 可以直接在电脑上烧录　　B. 需要借助 TF 卡系统中转烧录　　C. 不需要拨码开关　　D. 只能用 Win32Diskimager 烧录

### 二、判断题

**6.** balenaEtcher 可以直接烧录 `.img.xz` 压缩格式的镜像，无需手动解压。（　）

**7.** 开发板上有两个绿色 LED 灯，靠近关机按键的绿灯由软件控制其亮灭。（　）

**8.** 通过修改 `/etc/network/interfaces` 配置文件来连接 WIFI 是推荐的做法。（　）

**9.** 调试串口的波特率必须设置为 115200。（　）

**10.** Orange Pi AI Pro 搭载的昇腾 AI 处理器可提供 8 TOPS INT8 的计算能力。（　）

### 三、简答题

**11.** 为什么烧录 eMMC 和 SSD 不能像 TF 卡一样直接在电脑上完成，而需要先制作 TF 卡启动系统？

**12.** 切换拨码开关后为什么必须重新上下电才能生效，而按复位键无效？

**13.** 如何通过板载两个 LED 灯的状态初步判断开发板的上电与内核启动情况？

**14.** 在无法连接 HDMI 显示器的场合，可以通过哪几种方式登录开发板的 Linux 系统？

**15.** 简述 Orange Pi AI Pro 开发板三种启动介质（TF 卡、eMMC、SSD）的烧录流程差异。

> 📝 拓展题参考答案见 `answer/` 目录下的对应文件。


### 拨码开关启动设备查询器（可在云平台运行）

输入拨码开关状态，自动查询对应的启动设备：


In [ ]:
# 可在云平台运行 —— 拨码开关启动设备查询器
def query_boot_device(boot1, boot2):
    """根据 BOOT1/BOOT2 拨码开关状态查询启动设备"""
    boot_map = {
        ('左', '左'): '未使用',
        ('右', '左'): 'SATA SSD 和 NVMe SSD',
        ('左', '右'): 'eMMC',
        ('右', '右'): 'TF 卡',
    }
    key = (boot1, boot2)
    device = boot_map.get(key, '未知组合')
    
    print(f"┌──────────────────────────────┐")
    print(f"│  BOOT1 = {boot1}  BOOT2 = {boot2}        │")
    print(f"│  启动设备: {device}" + " " * max(0, 14 - len(device) - 4) + "│")
    print(f"└──────────────────────────────┘")
    return device

print('=== 拨码开关启动设备查询器 ===')
print()
print('本实验默认配置（TF 卡启动）:')
query_boot_device('右', '右')
print()
print('所有有效配置:')
for b1 in ['左', '右']:
    for b2 in ['左', '右']:
        query_boot_device(b1, b2)
        print()


---

## 八、实验总结

本次实验完成了 Orange Pi AI Pro（昇腾香橙派）开发板的系统制作烧录与基础运行验证。通过实验：

1. 掌握了三种启动介质（TF 卡、eMMC、SSD）的镜像烧录流程：TF 卡可直接在电脑上使用 Win32Diskimager 或 balenaEtcher 烧录，eMMC 和 SSD 则需借助 TF 卡系统中转烧录。
2. 理解了 BOOT1、BOOT2 拨码开关与启动设备的对应关系及其生效条件（必须重新上下电）。
3. 成功完成了开发板的上电启动、HDMI 桌面与调试串口登录。
4. 通过 LED 灯状态、以太网与 WIFI 连通性测试、SSH 远程登录等环节验证了系统的基本运行功能。

以上实践为后续基于昇腾 AI 处理器的推理应用开发（CANN、ATC 模型转换、Jupyter Lab 在线推理案例等）打好了环境基础。

---

## 参考资料

- 香橙派官网（镜像与工具下载）：`http://www.orangepi.cn`
- Orange Pi AI Pro 用户手册：随开发板资料包提供
- balenaEtcher 烧录工具：`https://www.balena.io/etcher/`
- SD Card Formatter：`https://www.sdcard.org/downloads/formatter/`
- Win32Diskimager：`http://sourceforge.net/projects/win32diskimager/files/Archive/`
- MobaXterm 终端工具：`https://mobaxterm.mobatek.net/`
- 昇腾开发者社区：`https://www.hiascend.com`
